In [112]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [113]:
import json
import logging

import pandas as pd

from general_functions.call_api_with_account_id import (
    call_api_with_accountId,
    send_to_innkeepr_api_paginated,
)
from general_functions.constants import return_api_url
from general_functions.return_workspace_ids import return_workspace_ids


In [114]:
def save_to_json(data, filename):
    from pathlib import Path

    output_path = Path(
        "SprintStories/EN-xx-Treatment-Performance-Conversion-Signals"
    ) / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Wrote {len(data)} rows to {output_path.resolve()}")


def get_conversion_goal_signal_id(usage_row: dict) -> str | None:
    conversion_goal = usage_row.get("conversionGoal") or {}
    signal_id = conversion_goal.get("signalId")
    if signal_id is None:
        return None
    text = str(signal_id).strip()
    return text or None


def match_usage_row_to_signal_by_conversion_goal(
    usage_row: dict,
    signal: dict,
) -> bool:
    usage_signal_id = get_conversion_goal_signal_id(usage_row)
    signal_id = signal.get("id")
    if not usage_signal_id or signal_id is None:
        return False
    return str(signal_id) == usage_signal_id


def match_signals_to_usage_by_conversion_goal(
    signals: list[dict],
    usage_rows: list[dict],
) -> pd.DataFrame:
    columns = [
        "signal.id",
        "signal.name",
        "conversionGoal.signalId",
        "conversionGoal.name",
        "usage.date",
        "usage.treatment",
        "usage.connectionId",
        "usage.connectionName",
        "usage.spend",
        "usage.impressions",
        "usage.clicks",
        "usage.conversions",
    ]
    if not signals or not usage_rows:
        return pd.DataFrame(columns=columns)

    matched_rows: list[dict] = []
    for signal in signals:
        signal_id = signal.get("id")
        signal_name = signal.get("name")

        for usage_row in usage_rows:
            if not match_usage_row_to_signal_by_conversion_goal(usage_row, signal):
                continue

            conversion_goal = usage_row.get("conversionGoal") or {}
            matched_rows.append(
                {
                    "signal.id": signal_id,
                    "signal.name": signal_name,
                    "conversionGoal.signalId": conversion_goal.get("signalId"),
                    "conversionGoal.name": conversion_goal.get("name"),
                    "usage.date": usage_row.get("date"),
                    "usage.treatment": usage_row.get("treatment"),
                    "usage.connectionId": usage_row.get("connectionId"),
                    "usage.connectionName": usage_row.get("connectionName"),
                    "usage.spend": usage_row.get("spend"),
                    "usage.impressions": usage_row.get("impressions"),
                    "usage.clicks": usage_row.get("clicks"),
                    "usage.conversions": usage_row.get("conversions"),
                }
            )

    if not matched_rows:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(matched_rows)


def get_treatment_external_id(treatment: dict) -> str | None:
    external_id = treatment.get("externalId")
    if external_id is None:
        return None
    text = str(external_id).strip()
    return text or None


def get_usage_treatment_external_id(usage_row: dict) -> str | None:
    usage_treatment = usage_row.get("treatment")
    if usage_treatment is None:
        return None
    text = str(usage_treatment).strip()
    if not text:
        return None
    if "/" not in text:
        return text
    return text.rsplit("/", 1)[-1]


def match_usage_row_to_treatment_by_external_id(
    usage_row: dict,
    treatment: dict,
) -> bool:
    treatment_external_id = get_treatment_external_id(treatment)
    usage_external_id = get_usage_treatment_external_id(usage_row)
    if not treatment_external_id or not usage_external_id:
        return False
    return treatment_external_id == usage_external_id


def get_treatment_campaign_name(treatment: dict) -> str | None:
    relates_to = treatment.get("relates_to") or {}
    campaign = relates_to.get("campaign") or {}
    name = campaign.get("name")
    return str(name) if name is not None else None


def build_treatment_lookup_by_external_id(treatments: list[dict]) -> dict[str, dict]:
    lookup: dict[str, dict] = {}
    for treatment in treatments:
        external_id = get_treatment_external_id(treatment)
        if external_id and external_id not in lookup:
            lookup[external_id] = treatment
    return lookup


def match_treatments_to_usage_by_external_id(
    treatments: list[dict],
    usage_rows: list[dict],
) -> pd.DataFrame:
    columns = [
        "treatment.id",
        "treatment.name",
        "campaign.name",
        "treatment.externalId",
        "usage.treatment",
        "usage.date",
        "usage.connectionId",
        "usage.connectionName",
        "usage.spend",
        "usage.impressions",
        "usage.clicks",
        "usage.conversions",
        "conversionGoal.name",
        "conversionGoal.signalId",
    ]
    if not treatments or not usage_rows:
        return pd.DataFrame(columns=columns)

    treatment_lookup = build_treatment_lookup_by_external_id(treatments)
    matched_rows: list[dict] = []

    for usage_row in usage_rows:
        usage_external_id = get_usage_treatment_external_id(usage_row)
        if not usage_external_id:
            continue

        treatment = treatment_lookup.get(usage_external_id)
        if treatment is None:
            continue

        conversion_goal = usage_row.get("conversionGoal") or {}
        conversion_goal_signal_id = conversion_goal.get("signalId")
        if not conversion_goal_signal_id:
            continue

        matched_rows.append(
            {
                "treatment.id": treatment.get("id"),
                "treatment.name": treatment.get("name"),
                "campaign.name": get_treatment_campaign_name(treatment),
                "treatment.externalId": get_treatment_external_id(treatment),
                "usage.treatment": usage_row.get("treatment"),
                "usage.date": usage_row.get("date"),
                "usage.connectionId": usage_row.get("connectionId"),
                "usage.connectionName": usage_row.get("connectionName"),
                "usage.spend": usage_row.get("spend"),
                "usage.impressions": usage_row.get("impressions"),
                "usage.clicks": usage_row.get("clicks"),
                "usage.conversions": usage_row.get("conversions"),
                "conversionGoal.name": conversion_goal.get("name"),
                "conversionGoal.signalId": conversion_goal_signal_id,
            }
        )

    if not matched_rows:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(matched_rows)


def build_signal_lookup_by_id(signals: list[dict]) -> dict[str, dict]:
    lookup: dict[str, dict] = {}
    for signal in signals:
        signal_id = signal.get("id")
        if signal_id is not None:
            lookup[str(signal_id)] = signal
    return lookup


def build_usage_treatment_signal_matches(
    usage_rows: list[dict],
    treatments: list[dict],
    conversion_signals: list[dict],
) -> pd.DataFrame:
    """Match usage log rows to treatments (externalId) and conversion signals (conversionGoal.signalId)."""
    columns = [
        "usage.date",
        "usage.treatment",
        "usage.connectionId",
        "usage.connectionName",
        "usage.spend",
        "usage.impressions",
        "usage.clicks",
        "usage.conversions",
        "treatment.id",
        "treatment.name",
        "campaign.name",
        "treatment.externalId",
        "conversionGoal.signalId",
        "conversionGoal.name",
        "signal.id",
        "signal.name",
        "signal.externalId",
    ]
    if not usage_rows:
        return pd.DataFrame(columns=columns)

    treatment_lookup = build_treatment_lookup_by_external_id(treatments)
    signal_lookup = build_signal_lookup_by_id(conversion_signals)
    matched_rows: list[dict] = []

    for usage_row in usage_rows:
        conversion_goal = usage_row.get("conversionGoal") or {}
        conversion_goal_signal_id = conversion_goal.get("signalId")
        if not conversion_goal_signal_id:
            continue

        usage_external_id = get_usage_treatment_external_id(usage_row)
        if not usage_external_id:
            continue

        treatment = treatment_lookup.get(usage_external_id)
        if treatment is None:
            continue

        signal = signal_lookup.get(str(conversion_goal_signal_id))
        if signal is None:
            continue

        matched_rows.append(
            {
                "usage.date": usage_row.get("date"),
                "usage.treatment": usage_row.get("treatment"),
                "usage.connectionId": usage_row.get("connectionId"),
                "usage.connectionName": usage_row.get("connectionName"),
                "usage.spend": usage_row.get("spend"),
                "usage.impressions": usage_row.get("impressions"),
                "usage.clicks": usage_row.get("clicks"),
                "usage.conversions": usage_row.get("conversions"),
                "treatment.id": treatment.get("id"),
                "treatment.name": treatment.get("name"),
                "campaign.name": get_treatment_campaign_name(treatment),
                "treatment.externalId": get_treatment_external_id(treatment),
                "conversionGoal.signalId": conversion_goal_signal_id,
                "conversionGoal.name": conversion_goal.get("name"),
                "signal.id": signal.get("id"),
                "signal.name": signal.get("name"),
                "signal.externalId": signal.get("externalId"),
            }
        )

    if not matched_rows:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(matched_rows)

In [115]:
customer = "to teach"  # change as needed
lookback_days = 3

url = return_api_url()
print(f"url = {url}")

workspaces = return_workspace_ids()
workspace_matches = [acc for acc in workspaces if acc["name"] == customer]
if len(workspace_matches) != 1:
    raise ValueError(
        f"Expected exactly one workspace named {customer!r}, found {len(workspace_matches)}. "
        f"Available: {sorted(acc['name'] for acc in workspaces)}"
    )
workspace_id = workspace_matches[0]["id"]

to_date = pd.Timestamp.today().normalize()
from_date = to_date - pd.Timedelta(days=lookback_days)
from_date_str = from_date.strftime("%Y%m%d")
to_date_str = to_date.strftime("%Y%m%d")

print(f"customer = {customer}")
print(f"workspace_id = {workspace_id}")
print(f"usage date range = {from_date_str} to {to_date_str} (last 3 days)")

# Load Data

In [116]:
conversion_signals = call_api_with_accountId(
    f"{url}api/signals/query",
    workspace_id,
    {"type": "conversion"},
    logging,
)
save_to_json(conversion_signals, f"{customer.replace(' ', '_')}_{from_date_str}_{to_date_str}_conversion_signals.json")
conversion_signals_df = pd.json_normalize(conversion_signals)
print(f"Found {len(conversion_signals_df)} conversion signals")
conversion_signals_df

In [117]:
usage_rows = send_to_innkeepr_api_paginated(
    f"{url}api/signals/usage/query",
    workspace_id,
    {"fromDate": from_date_str, "toDate": to_date_str},
    logging,
)

print(f"Found {len(usage_rows)} usage rows for {from_date_str} to {to_date_str}")


In [118]:
save_to_json(usage_rows, f"{customer.replace(' ', '_')}_{from_date_str}_{to_date_str}_usage_rows.json")

In [119]:
usage_rows_with_innkeepr = [
    row for row in usage_rows
    if "Innkeepr" in row.get("conversionGoal",{}).get("name", "")
]
max_date = max(row["date"] for row in usage_rows_with_innkeepr)
print(f"Max date: {max_date}")
save_to_json(usage_rows_with_innkeepr, f"{customer.replace(' ', '_')}_{from_date_str}_{to_date_str}_usage_rows_innkeepr.json")
usage_rows_with_innkeepr

In [120]:
connection_ids = [row["connectionId"] for row in usage_rows_with_innkeepr]
connection_ids = list(set(connection_ids))
treatments = send_to_innkeepr_api_paginated(
    f"{url}api/treatments/query",
    workspace_id,
    {"connectionIds": connection_ids},
    logging,
)
print(f"Found {len(treatments)} treatments")
save_to_json(treatments, f"{customer.replace(' ', '_')}_{from_date_str}_{to_date_str}_treatments.json")

# Matching Conversion Signal - Usage Log

In [121]:
signal_usage_matches = match_signals_to_usage_by_conversion_goal(
    conversion_signals,
    usage_rows,
)

print(f"Matched {len(signal_usage_matches)} usage rows to conversion signals")
signal_usage_matches

# Matching Treatment - Usage Log

In [122]:
treatment_usage_matches = match_treatments_to_usage_by_external_id(
    treatments,
    usage_rows,
)

print(
    f"Matched {len(treatment_usage_matches)} usage rows to treatments "
    f"with conversionGoal.signalId"
)
treatment_usage_matches

# Create a Pipeline
Treatments - usage log - audiences

# Full pipeline: usage log -> treatments -> conversion signals

In [123]:
usage_treatment_signal_matches = build_usage_treatment_signal_matches(
    usage_rows,
    treatments,
    conversion_signals,
)

print(
    f"Matched {len(usage_treatment_signal_matches)} usage rows through "
    f"usage log -> treatment -> conversion signal"
)
usage_treatment_signal_matches